In [ ]:
 pip install pandas numpy matplotlib seaborn scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
 
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
 
plt.style.use('seaborn-v0_8-whitegrid')
 
COLORS = {
    'pass'  : '#1D9E75',
    'fail'  : '#E24B4A',
    'blue'  : '#378ADD',
    'amber' : '#BA7517',
    'purple': '#7F77DD',
}
MODEL_COLORS = ['#BA7517', '#E24B4A', '#1D9E75', '#378ADD']
 
 

In [ ]:
def load_data(filepath='student-performance.csv'):
    try:
        # Try comma separator first (your file uses comma)
        df = pd.read_csv(filepath, sep=',')
        # If only 1 column was parsed, try semicolon
        if df.shape[1] == 1:
            df = pd.read_csv(filepath, sep=';')
    except FileNotFoundError:
        raise FileNotFoundError(
            f"\n[ERROR] '{filepath}' not found.\n"
            "Place student-performance.csv in the same folder as this script.\n"
        )
 
    df.columns = df.columns.str.strip().str.replace('"', '')
 
    print("=" * 60)
    print("   STUDENT PERFORMANCE PREDICTION SYSTEM")
    print("=" * 60)
    print(f"\n[1] Dataset loaded : {df.shape[0]} rows x {df.shape[1]} columns")
    print(f"    Columns        : {list(df.columns)}")
    print(f"\n    First 5 rows:")
    print(df.head().to_string())
    return df
 

In [ ]:
def preprocess(df):
    df = df.copy()
 
    # Strip quotes from string values
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].str.strip().str.replace('"', '')
 
    # Convert G1, G2, G3 to numeric (they may be strings in your file)
    for col in ['G1', 'G2', 'G3']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
 
    # Drop rows where G3 is missing
    before = len(df)
    df.dropna(subset=['G3'], inplace=True)
    print(f"\n[2] Dropped {before - len(df)} rows with missing G3")
 
    # Create binary target: G3 >= 10 → Pass (1), else Fail (0)
    df['Pass'] = (df['G3'] >= 10).astype(int)
 
    pass_count = df['Pass'].sum()
    fail_count = len(df) - pass_count
    print(f"\n[2] Target column 'Pass' created from G3:")
    print(f"    Pass (1) — G3 >= 10 : {pass_count} students ({pass_count/len(df)*100:.1f}%)")
    print(f"    Fail (0) — G3 <  10 : {fail_count} students ({fail_count/len(df)*100:.1f}%)")
 
    # Encode all categorical (object) columns using LabelEncoder
    le = LabelEncoder()
    cat_cols = df.select_dtypes(include='object').columns.tolist()
    print(f"\n[2] Encoding categorical columns: {cat_cols}")
    for col in cat_cols:
        df[col] = le.fit_transform(df[col].astype(str))
 
    # Check missing values
    missing = df.isnull().sum()
    if missing.sum() > 0:
        print(f"\n[2] Missing values found — filling with column median:")
        print(missing[missing > 0])
        df.fillna(df.median(numeric_only=True), inplace=True)
    else:
        print(f"\n[2] No missing values found.")
 
    # Drop G3 — it was used to create the target, keeping it would leak info
    df.drop(columns=['G3'], inplace=True)
 
    print(f"\n[2] Final shape after preprocessing: {df.shape}")
    return df
 

In [ ]:
def run_eda(df):
    print("\n[3] Running EDA — generating plots...")
 
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    fig.suptitle('Exploratory Data Analysis — Student Performance',
                 fontsize=14, fontweight='bold')
 
    # Plot 1 — Pass/Fail count
    counts = df['Pass'].value_counts().sort_index()
    bars = axes[0, 0].bar(
        ['Fail (0)', 'Pass (1)'],
        [counts.get(0, 0), counts.get(1, 0)],
        color=[COLORS['fail'], COLORS['pass']],
        edgecolor='white', width=0.5
    )
    axes[0, 0].set_title('Pass / Fail Distribution')
    axes[0, 0].set_ylabel('Number of Students')
    for bar in bars:
        h = bar.get_height()
        axes[0, 0].text(bar.get_x() + bar.get_width() / 2, h + 2,
                        str(int(h)), ha='center', fontsize=11)
 
    # Plot 2 — Study time vs Pass/Fail
    df_str = df.copy()
    df_str['Pass'] = df_str['Pass'].map({0: 'Fail', 1: 'Pass'})
    sns.countplot(data=df_str, x='studytime', hue='Pass', ax=axes[0, 1],
                  palette={'Fail': COLORS['fail'], 'Pass': COLORS['pass']},
                  hue_order=['Fail', 'Pass'])
    axes[0, 1].set_title('Study Time vs Pass/Fail')
    axes[0, 1].set_xlabel('Study Time (1=<2h  2=2-5h  3=5-10h  4=>10h)')
    axes[0, 1].legend(title='')
 
    # Plot 3 — Absences vs Pass/Fail
    sns.boxplot(data=df_str, x='Pass', y='absences', ax=axes[0, 2],
                palette={'Fail': COLORS['fail'], 'Pass': COLORS['pass']},
                order=['Fail', 'Pass'])
    axes[0, 2].set_title('Absences vs Pass/Fail')
    axes[0, 2].set_xlabel('')
 
    # Plot 4 — Past failures vs Pass/Fail
    sns.countplot(data=df_str, x='failures', hue='Pass', ax=axes[1, 0],
                  palette={'Fail': COLORS['fail'], 'Pass': COLORS['pass']},
                  hue_order=['Fail', 'Pass'])
    axes[1, 0].set_title('Past Failures vs Pass/Fail')
    axes[1, 0].set_xlabel('Number of Past Failures')
    axes[1, 0].legend(title='')
 
    # Plot 5 — G1 grade distribution by result
    for label, color, name in [(0, COLORS['fail'], 'Fail'),
                                (1, COLORS['pass'], 'Pass')]:
        subset = df[df['Pass'] == label]['G1']
        subset.plot(kind='kde', ax=axes[1, 1], color=color, label=name, linewidth=2)
    axes[1, 1].set_title('G1 (First Period Grade) Distribution')
    axes[1, 1].set_xlabel('G1 Grade (0–20)')
    axes[1, 1].legend()
 
    # Plot 6 — Top feature correlations with Pass
    corr = df.corr(numeric_only=True)[['Pass']]\
              .sort_values('Pass', ascending=False)\
              .drop('Pass')\
              .head(12)
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
                ax=axes[1, 2], cbar=True, linewidths=0.5)
    axes[1, 2].set_title('Top Feature Correlations with Pass')
    axes[1, 2].set_yticklabels(axes[1, 2].get_yticklabels(), rotation=0)
 
    plt.tight_layout()
    plt.savefig('01_eda.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("   Saved: 01_eda.png")

In [ ]:
def prepare_features(df):
    # Top features based on domain knowledge + correlation with Pass
    # G2 included because it's a strong predictor (mid-year grade)
    desired = ['G2', 'G1', 'failures', 'studytime', 'absences',
               'Medu', 'Fedu', 'goout', 'age', 'freetime',
               'health', 'romantic', 'Dalc', 'Walc', 'traveltime']
 
    # Keep only columns that actually exist in this dataset
    top_features = [f for f in desired if f in df.columns]
 
    X = df[top_features]
    y = df['Pass']
 
    print(f"\n[4] Features selected ({len(top_features)}): {top_features}")
    print(f"    X shape: {X.shape}  |  y shape: {y.shape}")
    print(f"    Class balance — Pass: {y.sum()}  Fail: {(y==0).sum()}")
 
    # Standardise features
    scaler   = StandardScaler()
    X_scaled = scaler.fit_transform(X)
 
    # 80/20 stratified split (stratify preserves class ratio)
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, random_state=42, stratify=y
    )
 
    print(f"\n[4] Train set: {X_train.shape[0]} samples")
    print(f"    Test  set: {X_test.shape[0]} samples")
    return X_train, X_test, y_train, y_test, scaler, top_features

In [ ]:
def train_models(X_train, X_test, y_train, y_test):
    models = {
        'Logistic Regression': LogisticRegression(
            random_state=42, max_iter=1000, C=1.0),
        'Decision Tree'      : DecisionTreeClassifier(
            random_state=42, max_depth=5, min_samples_split=5),
        'Random Forest'      : RandomForestClassifier(
            n_estimators=100, random_state=42, max_depth=10),
        'SVM'                : SVC(
            kernel='rbf', C=1.0, gamma='scale', random_state=42),
    }
 
    results = {}
    print("\n[5] Training all 4 models...")
    print(f"\n    {'Model':<25} {'Test Acc':>10}  {'CV Mean (5-fold)':>17}  {'CV Std':>8}")
    print("    " + "-" * 65)
 
    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        acc    = accuracy_score(y_test, y_pred)
        cv     = cross_val_score(model, X_train, y_train,
                                 cv=5, scoring='accuracy')
        results[name] = {
            'model'   : model,
            'y_pred'  : y_pred,
            'accuracy': acc,
            'cv_mean' : cv.mean(),
            'cv_std'  : cv.std(),
        }
        print(f"    {name:<25} {acc*100:>9.2f}%  "
              f"{cv.mean()*100:>15.2f}%  ±{cv.std()*100:.2f}%")
 
    best_name = max(results, key=lambda k: results[k]['accuracy'])
    print(f"\n    Best model : {best_name}")
    print(f"    Accuracy   : {results[best_name]['accuracy']*100:.2f}%")
    return results, best_name
 

In [ ]:
def evaluate_models(results, y_test, best_name):
    print(f"\n[6] Detailed report — {best_name}")
    print(classification_report(
        y_test, results[best_name]['y_pred'],
        target_names=['Fail', 'Pass']
    ))
 
    # Accuracy bar chart
    names  = list(results.keys())
    scores = [results[n]['accuracy'] * 100 for n in names]
 
    fig, ax = plt.subplots(figsize=(9, 4))
    bars = ax.barh(names, scores, color=MODEL_COLORS,
                   edgecolor='white', height=0.5)
    for bar, score in zip(bars, scores):
        ax.text(bar.get_width() + 0.3,
                bar.get_y() + bar.get_height() / 2,
                f'{score:.1f}%', va='center', fontsize=11)
    ax.set_xlabel('Accuracy (%)')
    ax.set_xlim(60, 100)
    ax.set_title('Model Accuracy Comparison', fontsize=13, fontweight='bold')
    ax.axvline(x=80, color='gray', linestyle='--',
               linewidth=1, label='80% reference')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig('02_model_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("   Saved: 02_model_comparison.png")
 
    # Confusion matrices — all 4 models
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    fig.suptitle('Confusion Matrices — All Models',
                 fontsize=13, fontweight='bold')
    for ax, (name, res) in zip(axes, results.items()):
        cm   = confusion_matrix(y_test, res['y_pred'])
        disp = ConfusionMatrixDisplay(cm, display_labels=['Fail', 'Pass'])
        disp.plot(ax=ax, colorbar=False, cmap='Blues')
        ax.set_title(f"{name}\nAcc: {res['accuracy']*100:.1f}%", fontsize=10)
    plt.tight_layout()
    plt.savefig('03_confusion_matrices.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("   Saved: 03_confusion_matrices.png")
 

In [ ]:
def plot_feature_importance(results, top_features):
    if 'Random Forest' not in results:
        return
 
    rf           = results['Random Forest']['model']
    importances  = pd.Series(rf.feature_importances_, index=top_features)
    importances  = importances.sort_values(ascending=True)
    median_val   = importances.median()
    bar_colors   = [COLORS['pass'] if v >= median_val else COLORS['blue']
                    for v in importances]
 
    fig, ax = plt.subplots(figsize=(9, 5))
    importances.plot(kind='barh', ax=ax, color=bar_colors, edgecolor='white')
    ax.axvline(x=median_val, color='gray', linestyle='--',
               linewidth=1.2, label=f'Median ({median_val:.3f})')
    ax.set_title('Feature Importance — Random Forest',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Importance Score')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig('04_feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("   Saved: 04_feature_importance.png")
 
    print("\n[7] Top 5 most important features:")
    top5 = importances.sort_values(ascending=False).head(5)
    for feat, score in top5.items():
        print(f"    {feat:<20} {score:.4f}")
 

In [ ]:
def print_tree_rules(results, top_features):
    if 'Decision Tree' not in results:
        return
    dt    = results['Decision Tree']['model']
    rules = export_text(dt, feature_names=list(top_features), max_depth=3)
    print("\n[8] Decision Tree Rules (top 3 levels):")
    print(rules[:2000])
 
 

In [ ]:
def identify_at_risk(filepath, results, best_name, scaler, top_features):
    best_model = results[best_name]['model']
 
    # Reload original raw data
    df_orig = pd.read_csv(filepath, sep=',')
    if df_orig.shape[1] == 1:
        df_orig = pd.read_csv(filepath, sep=';')
    df_orig.columns = df_orig.columns.str.strip().str.replace('"', '')
 
    # Strip quotes from string values
    for col in df_orig.select_dtypes(include='object').columns:
        df_orig[col] = df_orig[col].str.strip().str.replace('"', '')
 
    # Convert grades to numeric
    for col in ['G1', 'G2', 'G3']:
        if col in df_orig.columns:
            df_orig[col] = pd.to_numeric(df_orig[col], errors='coerce')
 
    df_orig.dropna(subset=['G3'], inplace=True)
    df_orig['Student_ID']   = range(1, len(df_orig) + 1)
    df_orig['Actual_Pass']  = (df_orig['G3'] >= 10).astype(int)
 
    # Encode categoricals
    le = LabelEncoder()
    for col in df_orig.select_dtypes(include='object').columns:
        df_orig[col] = le.fit_transform(df_orig[col].astype(str))
 
    df_orig.drop(columns=['G3'], inplace=True)
 
    # Only use features that exist
    available = [f for f in top_features if f in df_orig.columns]
    X_all = scaler.transform(df_orig[available])
 
    df_orig['Predicted_Pass'] = best_model.predict(X_all)
    df_orig['At_Risk']        = (df_orig['Predicted_Pass'] == 0).astype(int)
 
    at_risk = df_orig[df_orig['At_Risk'] == 1]
    print(f"\n[9] At-risk students: {len(at_risk)} out of {len(df_orig)} total")
 
    # Build output columns safely
    out_cols = ['Student_ID', 'At_Risk', 'Predicted_Pass', 'Actual_Pass']
    for col in ['studytime', 'absences', 'failures', 'G1', 'G2']:
        if col in df_orig.columns:
            out_cols.append(col)
 
    at_risk[out_cols].to_csv('at_risk_students.csv', index=False)
    print("   Saved: at_risk_students.csv")
    print("\n   Sample at-risk students:")
    print(at_risk[out_cols].head(10).to_string(index=False))
    return at_risk

In [ ]:
def print_summary(results, y_test):
    print("\n" + "=" * 65)
    print("   FINAL MODEL EVALUATION SUMMARY")
    print("=" * 65)
    print(f"\n  {'Model':<25} {'Test Acc':>10}  {'CV Mean':>10}  {'CV Std':>8}")
    print("  " + "-" * 58)
 
    for name, res in results.items():
        print(f"  {name:<25} {res['accuracy']*100:>9.2f}%"
              f"  {res['cv_mean']*100:>9.2f}%  ±{res['cv_std']*100:.2f}%")
 
    best = max(results, key=lambda k: results[k]['accuracy'])
    print(f"\n  Best Model : {best}")
    print(f"  Accuracy   : {results[best]['accuracy']*100:.2f}%")
 
    from sklearn.metrics import precision_score, recall_score, f1_score
    y_pred = results[best]['y_pred']
    print(f"  Precision  : {precision_score(y_test, y_pred)*100:.2f}%")
    print(f"  Recall     : {recall_score(y_test, y_pred)*100:.2f}%")
    print(f"  F1-Score   : {f1_score(y_test, y_pred)*100:.2f}%")
    print("=" * 65)
 

In [ ]:
def main():
    FILEPATH = 'student-performance.csv'
 
    # Step 1 — Load
    df = load_data(FILEPATH)
 
    # Step 2 — Preprocess
    df = preprocess(df)
 
    # Step 3 — EDA
    run_eda(df)
 
    # Step 4 — Features & split
    X_train, X_test, y_train, y_test, scaler, feats = prepare_features(df)
 
    # Step 5 — Train models
    results, best_name = train_models(X_train, X_test, y_train, y_test)
 
    # Step 6 — Evaluate
    evaluate_models(results, y_test, best_name)
 
    # Step 7 — Feature importance
    plot_feature_importance(results, feats)
 
    # Step 8 — Decision tree rules
    print_tree_rules(results, feats)
 
    # Step 9 — At-risk export
    identify_at_risk(FILEPATH, results, best_name, scaler, feats)
 
    # Step 10 — Final summary
    print_summary(results, y_test)
 
    print("\n" + "=" * 65)
    print("  DONE — all output files saved in this folder!")
    print("  Files: 01_eda.png, 02_model_comparison.png,")
    print("         03_confusion_matrices.png, 04_feature_importance.png,")
    print("         at_risk_students.csv")
    print("=" * 65)
 
 
if __name__ == '__main__':
    main()
 

In [13]:
# ============================================================
#   STUDENT PERFORMANCE — ACCURACY & ALL METRICS
#   Paste this as a NEW cell and run it directly
# ============================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score,
                             confusion_matrix, classification_report)

# ── 1. Load & preprocess ─────────────────────────────────────
df = pd.read_csv('student-performance.csv', sep=',')
df.columns = df.columns.str.strip().str.replace('"', '')

for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip().str.replace('"', '')

for col in ['G1', 'G2', 'G3']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df.dropna(subset=['G3'], inplace=True)

# Target column
df['Pass'] = (df['G3'] >= 10).astype(int)

# Encode categorical columns
le = LabelEncoder()
for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col].astype(str))

df.drop(columns=['G3'], inplace=True)

# ── 2. Features & split ──────────────────────────────────────
features = ['G2', 'G1', 'failures', 'studytime', 'absences',
            'Medu', 'Fedu', 'goout', 'age', 'freetime']
features = [f for f in features if f in df.columns]

X = df[features]
y = df['Pass']

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y)

# ── 3. Train all 4 models ────────────────────────────────────
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree'      : DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest'      : RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM'                : SVC(kernel='rbf', random_state=42)
}

# ── 4. Print all metrics ─────────────────────────────────────
print("=" * 65)
print("        STUDENT PERFORMANCE — MODEL ACCURACY REPORT")
print("=" * 65)
print(f"{'Model':<22} {'Accuracy':>9} {'Precision':>10} {'Recall':>8} {'F1':>8}")
print("-" * 65)

best_acc   = 0
best_name  = ''
best_pred  = None

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred, zero_division=0)
    f1   = f1_score(y_test, y_pred, zero_division=0)

    print(f"{name:<22} {acc*100:>8.2f}%  {prec*100:>8.2f}%  "
          f"{rec*100:>6.2f}%  {f1*100:>6.2f}%")

    if acc > best_acc:
        best_acc, best_name, best_pred = acc, name, y_pred

print("=" * 65)
print(f"\n  Best Model : {best_name}")
print(f"  Accuracy   : {best_acc*100:.2f}%")

# ── 5. Detailed report for best model ────────────────────────
print(f"\n  Detailed Classification Report — {best_name}")
print("-" * 65)
print(classification_report(y_test, best_pred, target_names=['Fail','Pass']))

# ── 6. Confusion matrix ──────────────────────────────────────
cm = confusion_matrix(y_test, best_pred)
tn, fp, fn, tp = cm.ravel()
print("  Confusion Matrix:")
print(f"  {'':15} Predicted Fail   Predicted Pass")
print(f"  Actual Fail  {'':4}  TN={tn:>3}            FP={fp:>3}")
print(f"  Actual Pass  {'':4}  FN={fn:>3}            TP={tp:>3}")
print(f"\n  Correctly predicted : {tp+tn} / {len(y_test)} students")
print(f"  Wrong predictions   : {fp+fn} / {len(y_test)} students")
print("=" * 65)

        STUDENT PERFORMANCE — MODEL ACCURACY REPORT
Model                   Accuracy  Precision   Recall       F1
-----------------------------------------------------------------
Logistic Regression       90.77%     94.55%   94.55%   94.55%
Decision Tree             86.92%     91.89%   92.73%   92.31%
Random Forest             90.77%     94.55%   94.55%   94.55%
SVM                       86.92%     91.89%   92.73%   92.31%

  Best Model : Logistic Regression
  Accuracy   : 90.77%

  Detailed Classification Report — Logistic Regression
-----------------------------------------------------------------
              precision    recall  f1-score   support

        Fail       0.70      0.70      0.70        20
        Pass       0.95      0.95      0.95       110

    accuracy                           0.91       130
   macro avg       0.82      0.82      0.82       130
weighted avg       0.91      0.91      0.91       130

  Confusion Matrix:
                  Predicted Fail   Predicted 